# 03. ML 전달 데이터 확인
최종 데이터 상태와 universe 정합성을 한 번에 확인합니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.validate import validate_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
final_df = pd.read_parquet(PROCESSED_DIR / 'final_df.parquet')
coverage_df = pd.read_csv(PROCESSED_DIR / 'coverage_df.csv', parse_dates=['actual_start_date', 'actual_end_date'])
final_missing_df = pd.read_csv(PROCESSED_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')

In [2]:
summary = {
 'final_df shape': final_df.shape, '최종 종목 수': final_df['Ticker'].nunique(),
 '전체 행 수': len(final_df), '날짜 범위': (final_df['Date'].min(), final_df['Date'].max()),
 '컬럼 목록': final_df.columns.tolist(), 'final_missing_df 종목 수': final_missing_df['Ticker'].nunique(),
 'short_history 종목 수': int(coverage_df['short_history'].sum()),
 'Ticker/Date 중복': int(final_df.duplicated(['Ticker','Date']).sum()),
 '필수 컬럼 결측': final_df[['Ticker','Date','Open','High','Low','Close','Volume']].isna().sum().to_dict()}
for key, value in summary.items(): print(f'{key}: {value}')
display(coverage_df.describe(include='all').T)
display(final_missing_df.groupby('fail_stage').size().rename('count'))

final_df shape: (1284440, 8)
최종 종목 수: 502
전체 행 수: 1284440
날짜 범위: (Timestamp('2016-01-04 00:00:00'), Timestamp('2026-06-30 00:00:00'))
컬럼 목록: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'source']
final_missing_df 종목 수: 1
short_history 종목 수: 36
Ticker/Date 중복: 0
필수 컬럼 결측: {'Ticker': 0, 'Date': 0, 'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Ticker,502,502,A,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_rows,502.0,NaN,NaN,NaN,2558.645418,24.0,2637.0,2637.0,2637.0,2637.0,336.823612
actual_start_date,502,NaN,NaN,NaN,2016-04-26 15:12:11.474103,2016-01-04 00:00:00,2016-01-04 00:00:00,2016-01-04 00:00:00,2016-01-04 00:00:00,2026-05-27 00:00:00,NaN
actual_end_date,502,NaN,NaN,NaN,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,NaN
n_price_imputed,502.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n_ohlc_inconsistent,502.0,NaN,NaN,NaN,0.539841,0.0,0.0,0.0,0.0,63.0,3.238584
n_volume_missing,502.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sources,502,2,yahoo,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
expected_rows_10y,502.0,NaN,NaN,NaN,2738.0,2738.0,2738.0,2738.0,2738.0,2738.0,0.0
coverage_10y,502.0,NaN,NaN,NaN,0.934484,0.0088,0.9631,0.9631,0.9631,0.9631,0.123016


fail_stage
collection    1
Name: count, dtype: int64

In [3]:
report = validate_ml_dataset(final_df, final_missing_df, sp500_universe, coverage_df)
print('[통과] 필수 검증 통과')
report

[통과] 필수 검증 통과


{'valid': True,
 'errors': [],
 'warnings': ['short_history 종목 36개(제외하지 않음)'],
 'final_shape': (1284440, 8),
 'n_final_tickers': 502,
 'n_missing_tickers': 1,
 'n_universe_tickers': 503}

short_history 종목 수: 36은 전체 분석 기간인 2016~2026년 데이터를 충분히 보유하지 않은 종목이 36개
short_history=True
actual_start_date > 2016-01-31 또는 10년 예상 거래일의 80% 미만만 보유
주로 다음과 같은 종목들입니다.
- 2016년 이후 신규 상장
- 기업 분할로 새로 생긴 종목
- 티커 변경·합병 이력이 있는 종목
- 일부 기간의 데이터가 제공되지 않는 종목

In [4]:
display(
    coverage_df.loc[
        coverage_df["short_history"],
        [
            "Ticker",
            "actual_start_date",
            "actual_end_date",
            "n_rows",
            "coverage_10y",
        ],
    ].sort_values("actual_start_date")
)

,Ticker,actual_start_date,actual_end_date,n_rows,coverage_10y
199,FTV,2016-07-05,2026-06-30,2511,0.9171
133,DELL,2016-08-17,2026-06-30,2480,0.9058
448,TTD,2016-09-21,2026-06-30,2456,0.8970
473,VST,2016-10-05,2026-06-30,2446,0.8934
237,HWM,2016-11-01,2026-06-30,2427,0.8864
247,INVH,2017-02-01,2026-06-30,2365,0.8638
123,CVNA,2017-04-28,2026-06-30,2305,0.8419
250,IR,2017-05-12,2026-06-30,2295,0.8382
465,VICI,2018-01-02,2026-06-30,2134,0.7794
471,VRT,2018-08-02,2026-06-30,1987,0.7257
